# Aula 5 — Dá para fazer melhor, e o que custa?

**Disciplina 2 · Aprendizado de Máquina · IASEG**

Na Aula 4 a turma chegou sozinha a uma conclusão: **a nota do teste depende do sorteio que separou o teste**.
O mesmo notebook, com outra divisão, contou outra história ("Soma 181" virou "Soma 3").

Hoje: o que fazer com isso, e o que custa tentar fazer melhor.

- **Parte 1** — O Titanic de novo
- **Parte 2** — Cada um com o seu sorteio, e a validação cruzada
- **Parte 3** — A floresta aleatória
- **Parte 4** — Comparar modelos direito
- **Parte 5** — Escolher a profundidade sem olhar o teste
- **Parte 6** — Vazamento: o número bom demais
- **No final** — a célula para o projeto de vocês

## Configuração

`pandas` para os dados, `numpy` para as contas. O resto é importado na etapa em que for usado.

In [1]:
import pandas as pd
import numpy as np

---

# Parte 1 — O Titanic de novo

A mesma preparação da Aula 4: as seis colunas do passageiro, sem os passageiros sem idade, e `sex` como 0/1.

In [2]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
COLUNAS = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']

df = pd.read_csv(URL)

# Só as colunas escolhidas e a resposta; tira quem não tem idade
dados = df[COLUNAS + ['survived']].dropna().copy()

# sex vira 0/1 (1 = mulher, 0 = homem)
dados['sex'] = (dados['sex'] == 'female').astype(int)

X = dados[COLUNAS]         # as entradas (features)
y = dados['survived']      # o alvo a prever (a classe)

print('Passageiros:', len(dados))

Passageiros: 714


## Etapa 3 — Divisão em treino e teste

**Usem exatamente estes valores: `test_size=0.3` e `random_state=42`.** Todo mundo precisa da mesma divisão,
para os números baterem entre vocês e com o slide (na Aula 4, uma cópia com outros valores contou outra história).

- `random_state=42` fixa o sorteio (qualquer número serviria; o importante é ser o mesmo para todos).
- `stratify=y` mantém a mesma proporção de sobreviventes no treino e no teste.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print('Treino:', len(X_train), 'passageiros | Teste:', len(X_test), 'passageiros')

Treino: 499 passageiros | Teste: 215 passageiros


### A referência

O chute que não aprende nada com as colunas: sempre a classe mais comum. É o chão de toda comparação de hoje.

In [4]:
from sklearn.dummy import DummyClassifier

referencia = DummyClassifier(strategy="most_frequent")
referencia.fit(X_train, y_train)

acc_ref = referencia.score(X_test, y_test)
print(f'Referência (sempre "não sobreviveu"): {acc_ref:.1%} no teste')

Referência (sempre "não sobreviveu"): 59.5% no teste


---

# Parte 2 — Cada um com o seu sorteio

Na Aula 4, uma divisão diferente mudou o resultado. Hoje a gente faz isso **de propósito**, a sala inteira.

Escolham **um número inteiro qualquer** (dia do aniversário, um número da sorte, o que quiserem) e escrevam
na célula abaixo. Ela separa o Titanic com o **seu** sorteio e treina a mesma árvore da Aula 4
(profundidade 5). Rodem e **digam o número do teste para o quadro**.

*(Esta célula usa nomes próprios, `meu_X_train` e companhia, para não mexer na divisão da Parte 1.)*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

meu_sorteio = 42    # o seu número: qualquer inteiro

meu_X_train, meu_X_test, meu_y_train, meu_y_test = train_test_split(X, y, test_size=0.3, random_state=meu_sorteio, stratify=y)

minha_arvore = DecisionTreeClassifier(max_depth=5, random_state=42)
minha_arvore.fit(meu_X_train, meu_y_train)

print(f'Sorteio {meu_sorteio}: teste {minha_arvore.score(meu_X_test, meu_y_test):.1%}')

Sorteio 1: teste 76.3%


Mesmo modelo, mesmos dados. Só o sorteio mudou, e a nota mudou junto. **Qual desses números é a nota "de verdade" da árvore?**

## A validação cruzada: cinco simulados em vez de uma prova

Na Aula 4: o treino é a lista de exercícios, o teste é a prova. Falta um meio-termo: **os simulados**.

1. Pegue **só o treino** e corte em 5 partes.
2. Treine em 4 partes e teste na 5ª.
3. Repita até cada parte ter sido o teste uma vez.

Cada passageiro do treino é testado **exatamente uma vez**, e o resultado são **cinco notas**, não uma.
O teste de verdade continua guardado: ele é a prova final, feita uma vez só.

**Antes de rodar:** de pessoa para pessoa na sala, a **média dos cinco simulados** vai variar mais, ou menos, que o teste?

In [27]:
from sklearn.model_selection import cross_val_score

# Os cinco simulados, no SEU treino (o teste não entra aqui)
notas = cross_val_score(minha_arvore, meu_X_train, meu_y_train, cv=5)

print('As cinco partes:', [f'{n:.1%}' for n in notas])
print(f'Sorteio {meu_sorteio}: média dos 5 simulados {notas.mean():.1%}')

As cinco partes: ['81.0%', '80.0%', '79.0%', '78.0%', '77.8%']
Sorteio 1: média dos 5 simulados 79.2%


**Digam a média para o quadro**, na segunda coluna.

A média de cinco provas balança menos que uma prova só. E as cinco notas mostram outra coisa:
**o quanto o resultado depende de quais passageiros caíram no teste** (a pergunta 11 do banco).

### Agora todos na mesma divisão

A mesma conta, na divisão da Parte 1: todo mundo vê os mesmos cinco números.

In [29]:
arvore5 = DecisionTreeClassifier(max_depth=5, random_state=42)

notas5 = cross_val_score(arvore5, X_train, y_train, cv=5)

print('As cinco partes:', [f'{n:.1%}' for n in notas5])
print(f'Média {notas5.mean():.1%} | pior parte {notas5.min():.1%} | melhor parte {notas5.max():.1%}')

As cinco partes: ['84.0%', '75.0%', '89.0%', '76.0%', '74.7%']
Média 79.7% | pior parte 74.7% | melhor parte 89.0%


In [30]:
# E a prova final, uma vez só
arvore5.fit(X_train, y_train)
print(f'Teste: {arvore5.score(X_test, y_test):.1%}')

Teste: 82.8%


As cinco partes vão de 74,7% a 89,0%, e a média é 79,7%. O teste da Aula 4, 82,8%, caiu perto da ponta de cima:
**foi um sorteio bom**, não a nota da árvore.

**Na prática:** `cv=5`; reportar a média **e** a pior e a melhor parte; o teste fica para o fim.

---

# Parte 3 — A floresta aleatória

## De volta à árvore sem limite

Na Aula 4, a árvore sem limite de profundidade acertou quase todo o treino e caiu no teste.
Olhem **quantos passageiros do treino caem em cada folha** dela:

In [ ]:
arvore_livre = DecisionTreeClassifier(random_state=42)   # sem max_depth
arvore_livre.fit(X_train, y_train)


# Quantos passageiros do treino caem em cada folha
por_folha = pd.Series(arvore_livre.apply(X_train)).value_counts()

print('Folhas:', arvore_livre.get_n_leaves())
print('Folhas com um passageiro só:', (por_folha == 1).sum())
print(f'Treino {arvore_livre.score(X_train, y_train):.1%} | teste {arvore_livre.score(X_test, y_test):.1%}')

Folhas: 109
Folhas com um passageiro só: 49
Treino 99.4% | teste 74.9%


Quase metade das folhas existe **por causa de um passageiro só**. A árvore ficou específica demais para o treino.

Dois jeitos de sair disso:

1. **Deixar a árvore mais simples:** limitar a profundidade (a Aula 4).
2. **Manter as árvores específicas, mas fazer muitas**, cada uma treinada num **sorteio diferente** dos passageiros, e colocar todas para **votar**.
   Isso é a **floresta aleatória** (*random forest*).

Por que a votação funciona:

- **O padrão aparece em todo sorteio** (mulheres de 1ª e 2ª classe sobreviveram): toda árvore aprende, e as árvores votam juntas.
- **A folha feita para um passageiro esquisito** só existe nas árvores que sortearam aquele passageiro, e cada uma cai num lugar diferente.
  Na votação, esses exageros discordam entre si e perdem.

*Como testemunhas de um mesmo fato: cada uma viu de um ângulo e lembra alguns detalhes errado, cada uma do seu jeito.
Aquilo em que todas concordam é o que dá para confiar.*

In [32]:
from sklearn.ensemble import RandomForestClassifier

# 100 árvores, cada uma num sorteio diferente dos passageiros do treino
floresta = RandomForestClassifier(n_estimators=100, random_state=42)
floresta.fit(X_train, y_train)

print(f'Floresta: treino {floresta.score(X_train, y_train):.1%} | teste {floresta.score(X_test, y_test):.1%}')
print(f'Árvore sem limite: treino {arvore_livre.score(X_train, y_train):.1%} | teste {arvore_livre.score(X_test, y_test):.1%}')

Floresta: treino 99.4% | teste 79.5%
Árvore sem limite: treino 99.4% | teste 74.9%


**O mesmo treino, e um teste bem melhor.**

Cada árvore da floresta, sozinha, vai bem no teste? `floresta.estimators_` é a lista das 100 árvores.

In [ ]:
# A nota de cada árvore sozinha no teste
# (.values: as árvores internas foram treinadas sem os nomes das colunas)
sozinhas = [arvore.score(X_test.values, y_test.values) for arvore in floresta.estimators_]

print(f'Cada árvore sozinha: média {np.mean(sozinhas):.1%} | pior {min(sozinhas):.1%} | melhor {max(sozinhas):.1%}')
print(f'A votação das 100:   {floresta.score(X_test, y_test):.1%}')

**Nenhuma das 100 árvores, sozinha, chega na nota da votação.**

### Como a floresta responde: uma votação

A passageira da Aula 4: mulher, 1ª classe, 31 anos. Quantas árvores dizem que ela sobreviveu?

In [ ]:
# A primeira mulher de 1ª classe do teste (a mesma da Aula 4)
passageira = X_test[(X_test['sex'] == 1) & (X_test['pclass'] == 1)].iloc[[0]]

votos = sum(arvore.predict(passageira.values)[0] for arvore in floresta.estimators_)

print(f'{int(votos)} de 100 árvores dizem "sobreviveu"')
print('A floresta responde:', 'sobreviveu' if floresta.predict(passageira)[0] == 1 else 'não sobreviveu')
print('De verdade:', 'sobreviveu' if y_test.loc[passageira.index[0]] == 1 else 'não sobreviveu')

Como a folha da árvore guardava uma proporção e cortava em 50%, a floresta conta os votos e corta em 50%.

### O treino não diz nada sobre a floresta

No treino, a floresta acerta quase tudo **por construção**: cada árvore acerta os passageiros que sorteou.
Por isso a nota do treino não serve para julgar uma floresta; **só dados que ela não viu** servem
(o teste, ou os simulados).

### Um hiperparâmetro novo: quantas árvores

Os **hiperparâmetros** são o que a gente escolhe antes de treinar (a profundidade, o *k* do kNN, o número de árvores).
Os **parâmetros** são o que o modelo aprende sozinho no treino (os pesos da regressão, as perguntas da árvore).

**Antes de rodar:** com mais árvores, o treino sobe? E o teste?

In [ ]:
linhas = []
for n in [1, 5, 10, 50, 100, 300]:
    f = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    linhas.append({'árvores': n,
                   'treino': round(100 * f.score(X_train, y_train), 1),
                   'teste': round(100 * f.score(X_test, y_test), 1)})

pd.DataFrame(linhas).set_index('árvores')

Mais árvores **não fazem a floresta decorar mais**: depois de umas 50, o teste para de mudar e só o tempo de treino cresce.
Os outros hiperparâmetros (`max_depth`, `min_samples_leaf`) são os mesmos da árvore, aplicados a cada uma.

**E o custo?** A árvore de profundidade 2 da Aula 4 tinha 4 folhas, e dava para ler em voz alta.

In [ ]:
print('Folhas somadas nas 100 árvores:', sum(arvore.get_n_leaves() for arvore in floresta.estimators_))

Ninguém lê isso. **A floresta não se lê**: dá para perguntar depois em que ela se apoiou (Aula 7), mas não dá para seguir um caminho.

---

# Parte 4 — Comparar modelos direito

Três regras:

1. **A mesma divisão** para todos os modelos.
2. **A mesma métrica** para todos.
3. **A mesma referência**, como primeira linha.

E uma regra nova: **escolher olhando os simulados, e só depois ver a prova.**

### A função da comparação

Uma função que recebe vários modelos e devolve **uma tabela**: treino, validação cruzada (média, pior e melhor parte) e,
**se vocês pedirem**, o teste. No final do notebook tem uma versão dela para colar no projeto de vocês.

In [ ]:
from sklearn.model_selection import cross_validate

def tabela_comparacao(modelos, X_train, y_train, X_test=None, y_test=None, metrica='accuracy', partes=5):
    """Uma linha por modelo: treino, validação cruzada (média, pior, melhor) e, se pedido, o teste."""
    from sklearn.metrics import get_scorer
    avaliador = get_scorer(metrica) if isinstance(metrica, str) else metrica
    # métricas de erro vêm negativas no scikit-learn ('neg_...'): desvira o sinal
    sinal = -1 if isinstance(metrica, str) and metrica.startswith('neg_') else 1
    # métricas de acerto viram porcentagem; métricas de erro ficam na unidade do alvo
    escala = 1 if sinal == -1 else 100

    linhas = []
    for nome, modelo in modelos.items():
        vc = sinal * escala * cross_validate(modelo, X_train, y_train, cv=partes, scoring=avaliador)['test_score']
        modelo.fit(X_train, y_train)
        linha = {'modelo': nome,
                 'treino': sinal * escala * avaliador(modelo, X_train, y_train),
                 'validação cruzada (média)': vc.mean(),
                 'pior parte': vc.max() if sinal == -1 else vc.min(),     # num erro, pior = maior
                 'melhor parte': vc.min() if sinal == -1 else vc.max()}
        if X_test is not None:
            linha['teste'] = sinal * escala * avaliador(modelo, X_test, y_test)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index('modelo').round(1 if escala == 100 else 2)

### Os modelos

A regressão logística e o kNN precisam das colunas na mesma escala (Aula 4). Aqui a escala vai **dentro do modelo**,
com `make_pipeline`: em cada simulado, a escala é aprendida só nas partes de treino daquele simulado.
É o vazamento da escala, da Aula 4, resolvido.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

modelos = {
    'Referência': DummyClassifier(strategy='most_frequent'),
    'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression()),
    'kNN (k = 5)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Árvore, prof. 3': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Floresta': RandomForestClassifier(n_estimators=100, random_state=42),
}

# Primeiro, SEM o teste
tabela_comparacao(modelos, X_train, y_train)

### Escolham agora, antes de ver o teste

Em dupla: qual modelo vocês escolheriam, e **por quê**? O motivo não precisa ser o maior número:
legibilidade, simplicidade ou rapidez também valem, desde que estejam escritos.

In [ ]:
escolha = '___'
motivo = '___'

print('Escolhemos:', escolha)
print('Porque:', motivo)

### Agora, a prova: o teste, uma vez

In [ ]:
tabela_comparacao(modelos, X_train, y_train, X_test, y_test)

**Lendo a tabela:**

- **Nos quatro modelos, o teste ficou de 1 a 3 pontos abaixo da validação cruzada.** Isso é normal, não é fracasso. O número que vai para o relatório é o do teste.
- Os quatro modelos ficaram a menos de 2,5 pontos uns dos outros na validação cruzada, e as partes de cada um variam de 11 a 17 pontos.
  **A diferença entre os modelos é menor que a diferença entre os simulados.**
- Por isso o "por quê" da escolha pode ser outra coisa que não o maior número.
- Aqui, escolher olhando o teste teria dado a mesma floresta. O problema de olhar o teste aparece **na média**, não em todo caso:
  na Aula 4, escolher o *k* do kNN olhando um conjunto guardado deu 83,2% na escolha e 79,9% em dados novos, **otimista em 16 de 20 sorteios**.

---

# Parte 5 — Escolher a profundidade sem olhar o teste

Na Aula 4, a "melhor profundidade" do Titanic foi escolhida olhando o teste. Agora: **escolher pelos simulados, e fazer a prova uma vez.**

In [ ]:
linhas = []
for profundidade in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, None]:
    notas_p = cross_val_score(DecisionTreeClassifier(max_depth=profundidade, random_state=42),
                              X_train, y_train, cv=5)
    linhas.append({'profundidade': 'sem limite' if profundidade is None else profundidade,
                   'validação cruzada (média)': round(100 * notas_p.mean(), 1),
                   'pior parte': round(100 * notas_p.min(), 1),
                   'melhor parte': round(100 * notas_p.max(), 1)})

pd.DataFrame(linhas).set_index('profundidade')

As profundidades de 2 a 5 praticamente empatam. A maior média é a 4; a regra da Aula 4, **no empate, a mais simples**, também vale.

Escolham a profundidade **olhando só esta tabela**, e só então rodem o teste.

In [ ]:
profundidade_escolhida = ___     # olhando só a tabela acima

arvore_final = DecisionTreeClassifier(max_depth=profundidade_escolhida, random_state=42)
arvore_final.fit(X_train, y_train)

print(f'Profundidade {profundidade_escolhida}: teste {arvore_final.score(X_test, y_test):.1%} (uma vez)')

**Para que isso serve**, com números (20 sorteios diferentes da divisão): escolhendo a profundidade **olhando o teste**,
o número reportado dá em média 81,6%; escolhendo **pela validação cruzada**, o teste intocado da árvore escolhida dá 79,5%.
Uns **2 pontos do primeiro número foram emprestados do juiz**.

A validação cruzada não acha uma árvore melhor. Ela **mantém honesto o número final**.

---

# Parte 6 — Vazamento: o número bom demais

## De volta ao banco

41.188 pessoas para quem o banco ligou; a coluna `y` diz quem aceitou o investimento.
O modelo da Aula 3 (as seis colunas que descrevem a pessoa) disse "não" para todo mundo.

**A pergunta de hoje:** o banco quer decidir **para quem ligar amanhã**. Para cada coluna: **o banco sabe isso antes de ligar?**

| Grupo | Coluna | O que é | Sabe antes de ligar? |
|---|---|---|---|
| a pessoa | `age`, `job`, `marital`, `education` | idade, profissão, estado civil, escolaridade | |
| | `default`, `housing`, `loan` | inadimplência, financiamento, empréstimo | |
| o contato | `contact` | celular ou fixo | |
| | `month`, `day_of_week` | mês e dia da semana da ligação | |
| | `duration` | quanto tempo a ligação durou, em segundos | |
| a campanha | `campaign` | quantas vezes a pessoa foi contatada nesta campanha, contando esta ligação | |
| | `pdays`, `previous`, `poutcome` | contatos e resultado de campanhas anteriores | |
| a economia | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | indicadores da economia no mês | |
| a resposta | `y` | aceitou ou não | |

### O modelo da Aula 3, de novo

A mesma preparação e a mesma divisão da Aula 3 (80/20, `random_state=42`, `stratify`).

In [ ]:
URL_BANCO = "https://raw.githubusercontent.com/HegdeChaitra/Bank-Marketing-Campaign-Analysis/master/bank-additional-full.csv"

banco = pd.read_csv(URL_BANCO, sep=';')

# As seis colunas da pessoa, em 0 e 1 (Aula 3), sem a coluna repetida
PESSOA = ['job', 'marital', 'education', 'default', 'housing', 'loan']
X_pessoa = pd.get_dummies(banco[PESSOA], drop_first=True).drop(columns='loan_unknown')
y_banco = banco['y']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_pessoa, y_banco, test_size=0.2, random_state=42, stratify=y_banco)

print('Pessoas no teste:', len(yb_test), '| aceitaram:', (yb_test == 'yes').sum())

In [ ]:
def quantos_achou(y_real, y_previsto):
    """Em contagem: para quantos o modelo disse 'sim', e quantos dos que aceitaram ele achou."""
    disse_sim = (y_previsto == 'yes').sum()
    achou = ((y_previsto == 'yes') & (y_real == 'yes')).sum()
    print(f'Disse "sim" para {disse_sim} pessoas | achou {achou} de {(y_real == "yes").sum()} que aceitaram'
          f' | acurácia {(y_previsto == y_real).mean():.1%}')

logistica_pessoa = LogisticRegression()
logistica_pessoa.fit(Xb_train, yb_train)

quantos_achou(yb_test, logistica_pessoa.predict(Xb_test))

Zero. **Antes de rodar:** e se a gente juntar **uma** coluna, `duration`?

In [ ]:
# As mesmas colunas, mais a duração da ligação
X_com_duracao = X_pessoa.copy()
X_com_duracao['duration'] = banco['duration']

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_com_duracao, y_banco, test_size=0.2, random_state=42, stratify=y_banco)

# duration está em segundos: a logística precisa da escala (Aula 4)
logistica_duracao = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logistica_duracao.fit(Xd_train, yd_train)

quantos_achou(yd_test, logistica_duracao.predict(Xd_test))

De nada para quase 200. A acurácia quase não se mexeu (a Aula 3 já mostrou que ela andava na direção errada aqui),
mas **a contagem saiu do zero**. Parece o melhor modelo do dia.

### Por que funciona

De cada 100 ligações, quantas terminaram em "sim", conforme a duração:

In [ ]:
faixas = pd.cut(banco['duration'], bins=[0, 60, 120, 300, 600, 5000], right=False,
                labels=['menos de 1 min', '1 a 2 min', '2 a 5 min', '5 a 10 min', 'mais de 10 min'])

pd.DataFrame({
    'ligações': banco.groupby(faixas, observed=True)['y'].size(),
    '"sim"': banco.groupby(faixas, observed=True)['y'].apply(lambda s: (s == 'yes').sum()),
    '"sim" de cada 100': banco.groupby(faixas, observed=True)['y'].apply(lambda s: round(100 * (s == 'yes').mean(), 1)),
})

Das 4.181 ligações com menos de um minuto, **uma** terminou em "sim". Das que passaram de dez minutos, quase metade.

**A duração é o jeito como a ligação foi.** Para saber a duração, o banco já precisa ter ligado. O modelo está
"prevendo" o resultado da ligação usando o resultado da ligação. No dia de decidir para quem ligar amanhã, essa coluna não existe.

E o jeito mais comum de cair nisso não é escolher `duration` de propósito: é **treinar com todas as colunas da base**,
e ela entra junto.

**A decisão:** que coluna sai?

In [ ]:
X_honesto = X_com_duracao.drop(columns=___)

print('Colunas:', X_honesto.shape[1], '| iguais às da Aula 3?', list(X_honesto.columns) == list(X_pessoa.columns))

### As formas de vazamento

| Forma | Exemplo |
|---|---|
| Uma coluna que é a resposta escrita de novo | `alive` no Titanic (Aula 4) |
| Uma coluna que só existe **depois** do fato | `duration` no banco |
| Algo aprendido com **todos** os dados antes de dividir | a escala; preencher vazios com a média da base inteira (Aula 4) |
| O teste usado para escolher | a Parte 5 |

**Na prática:** para cada coluna, perguntem **"no momento da decisão, eu teria essa informação?"** (pergunta 9 do banco).
Um resultado bom demais é motivo para procurar vazamento, não para comemorar.

---

# Fechando

**Dá para fazer melhor, e o que custa?**

1. **A leitura.** No Titanic, a floresta empata com uma árvore rasa, e custa todas as folhas que dava para ler.
2. **Um número melhor que é sorte.** Um sorteio só; a validação cruzada mostra a faixa.
3. **Um número melhor que é trapaça.** Escolher olhando o teste, ou uma coluna que vaza.

### Para o projeto

- **Seção 3:** a tabela de comparação (a célula abaixo), com a referência, os modelos testados, o escolhido e o porquê. Escolham antes de olhar o teste.
- **Seção 5:** o quanto o resultado depende da divisão (a pior e a melhor parte), e a caça ao vazamento nas colunas de vocês.
- Perguntas do banco: **9** (coluna que só existe depois), **10** (o que se perde com um modelo mais complexo), **11** (e se a divisão fosse outra).

---

# A célula para o projeto de vocês

Copiem a célula abaixo para o notebook de vocês **depois** da divisão treino/teste. Ela traz a mesma função da Parte 4.

1. Troquem os modelos do dicionário `modelos` pelos que vocês testaram (a referência sempre na primeira linha).
2. Rodem **sem o teste**, escolham e escrevam o porquê.
3. Só então rodem a última linha, **com o teste**.

- **Classificação:** `metrica='accuracy'`, ou a métrica que vocês escolheram na Seção 3.
  Se o alvo for texto (por exemplo `'Yes'`/`'No'`) e a métrica for recall, use
  `metrica=make_scorer(recall_score, pos_label='Yes')`.
- **Regressão:** `metrica='neg_mean_absolute_error'` (o erro médio da Aula 3), a referência `DummyRegressor()`, e
  `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`. O exemplo está na célula seguinte.
- **Base grande** (dezenas de milhares de linhas): a floresta demora; use `RandomForestClassifier(n_estimators=100, n_jobs=-1)`.

In [ ]:
# ===================== Tabela de comparação (Seção 3 do relatório) =====================
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.metrics import get_scorer, make_scorer, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

def tabela_comparacao(modelos, X_train, y_train, X_test=None, y_test=None, metrica='accuracy', partes=5):
    """Uma linha por modelo: treino, validação cruzada (média, pior, melhor) e, se pedido, o teste."""
    avaliador = get_scorer(metrica) if isinstance(metrica, str) else metrica
    sinal = -1 if isinstance(metrica, str) and metrica.startswith('neg_') else 1   # erros vêm negativos
    escala = 1 if sinal == -1 else 100                                          # acertos em %
    linhas = []
    for nome, modelo in modelos.items():
        vc = sinal * escala * cross_validate(modelo, X_train, y_train, cv=partes, scoring=avaliador)['test_score']
        modelo.fit(X_train, y_train)
        linha = {'modelo': nome,
                 'treino': sinal * escala * avaliador(modelo, X_train, y_train),
                 'validação cruzada (média)': vc.mean(),
                 'pior parte': vc.max() if sinal == -1 else vc.min(),     # num erro, pior = maior
                 'melhor parte': vc.min() if sinal == -1 else vc.max()}
        if X_test is not None:
            linha['teste'] = sinal * escala * avaliador(modelo, X_test, y_test)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index('modelo').round(1 if escala == 100 else 2)

# ---- os modelos de vocês (classificação; para regressão, veja a próxima célula) ----
modelos = {
    'Referência': DummyClassifier(strategy='most_frequent'),
    'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Árvore, prof. 3': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Floresta': RandomForestClassifier(n_estimators=100, random_state=42),
}

# 1) sem o teste: escolham olhando esta tabela
display(tabela_comparacao(modelos, X_train, y_train, metrica='accuracy'))

# 2) depois de escolher e escrever o porquê, descomentem a linha abaixo (o teste, uma vez)
# display(tabela_comparacao(modelos, X_train, y_train, X_test, y_test, metrica='accuracy'))

### Exemplo para quem faz regressão: a idade no banco (Aula 3)

A mesma pergunta da Aula 3 (*dá para prever a idade a partir do perfil?*), com a mesma divisão (80/20, `random_state=42`).
A métrica é o **erro médio, em anos**: aqui, **quanto menor, melhor**.

In [ ]:
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_pessoa, banco['age'], test_size=0.2, random_state=42)

modelos_regressao = {
    'Referência (sempre a média)': DummyRegressor(strategy='mean'),
    'Regressão linear': LinearRegression(),
    'Árvore, prof. 5': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Floresta': RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1),
}

tabela_comparacao(modelos_regressao, Xi_train, yi_train, Xi_test, yi_test, metrica='neg_mean_absolute_error')

O mesmo recado da Parte 4, em regressão: a floresta erra quase o mesmo que a regressão linear da Aula 3, e a regressão linear se lê.